# SKRIPSI: Train YOLOv12

---

Notebook ini dicopy dari [`How to Train YOLO11 Object Detection on a Custom Dataset`](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolo11-object-detection-on-custom-dataset.ipynb) dari Github Roboflow.  
  

*   Notebook ini dibuat untuk membangun model pendeteksi tanda vital di monitor fisiologis/bedside monitor.
*   Menggunakan pre-trained model **`YOLOv12`**
*   Dataset: [Dataset Skripsi_V1](https://app.roboflow.com/skripsian-bwueb/skripsi_v1/2/)

**NOTE: PASTIKAN ANDA MEMILIKI AKUN ROBOFLOW UNTUK MENGAKSES DAN MENGGUNAKAN DATASET**

## **STEP 1: Environment and Library Setup**

### 1.1 - Configure API keys
- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Kaggle, go to the `Add-ons` and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### 1.2 - Configure GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Settings` -> `Accelerator`, set it to `GPU P100`, and then click `Save`.

In [1]:
!nvidia-smi

Wed Apr 15 08:02:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.3 - Create a `HOME` constant.

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/kaggle/working


### 1.4 - Install Dependencies

In [3]:
!pip install ultralytics supervision roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6842.1/8062.4 GB disk)


## **STEP 2: Download Dataset via Code from Roboflow**

### 2.1 - Get datasets from Roboflow

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_API_KEY")

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("skripsian-bwueb").project("skripsi_v1")
version = project.version(2)
dataset = version.download("yolov12")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Skripsi_V1-2 in yolov12:: 100%|██████████| 3135/3135 [00:00<00:00, 8513.70it/s]


In [5]:
os.chdir("/kaggle/working/Skripsi_V1-2")
DATASET_PATH = os.getcwd()
print(DATASET_PATH)

/kaggle/working/Skripsi_V1-2


## **STEP 3: Start Custom Training!**

### 3.1 - Train with Code

In [6]:
!pip install wandb -q

In [7]:
# Setup wandb for monitoring
import wandb
from kaggle_secrets import UserSecretsClient

# Mengambil API Key dari Kaggle Secrets secara aman
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

# Login ke W&B
wandb.login(key=wandb_api)

# Inisialisasi proyek di W&B
wandb.init(project="skripsi-monitor-medis", name="yolov12n_baseline")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aizarhafizh (aizarhafizh-universitas-gadjah-mada-library) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run z4twuy8u
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/Skripsi_V1-2/wandb/run-20260415_080246-z4twuy8u
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run yolov12n_baseline
wandb: ⭐️ View project at https://wandb.ai/aizarhafizh-universitas-gadjah-mada-library/skripsi-monitor-medis
wandb: 🚀 View run at https://wandb.ai/aizarhafizh-univ

In [8]:
os.chdir(HOME)

# Pastikan Anda sudah menginstall versi terbaru: pip install ultralytics
from ultralytics import YOLO

# 1. Inisialisasi Model
# Gunakan arsitektur Nano untuk eksperimen awal yang ringan
model = YOLO('yolo12n.pt') 

# 2. Path ke file konfigurasi dataset
# Jika di Kaggle, sesuaikan path ini dengan folder hasil download Roboflow Anda
dataset_yaml = os.path.join(DATASET_PATH, "data.yaml")

# 3. Eksekusi Pelatihan dengan Konfigurasi Hyperparameter Lengkap
results = model.train(
    # ==========================================
    # A. PARAMETER DASAR (BASIC CONFIGURATION)
    # ==========================================
    data=dataset_yaml,
    epochs=100,             # Total iterasi pelatihan (100 sudah cukup untuk 1500+ data)
    imgsz=640,              # Ukuran resolusi gambar input (standar YOLO)
    batch=16,               # Jumlah gambar per iterasi (naikkan ke 32 jika memori GPU besar)
    device=[0,1],               # Gunakan GPU 0 (Kaggle). Jika pakai 2 GPU T4, ketik list: [0,1]
    workers=8,              # Jumlah thread CPU untuk memuat data (8 aman untuk Kaggle)
    project='runs/train',   # Direktori utama penyimpanan hasil
    name='baseline_yolov12n', # Nama folder eksperimen saat ini
    
    # ==========================================
    # B. PARAMETER OPTIMASI & LEARNING (TUNABLE)
    # ==========================================
    optimizer='auto',       # Pilihan: 'SGD', 'Adam', 'AdamW'. 'auto' sangat disarankan.
    lr0=0.01,               # Initial Learning Rate (Standar AdamW: 0.001, SGD: 0.01)
    lrf=0.01,               # Final Learning Rate (fraksi dari lr0, misal: 0.01 * lr0)
    # momentum=0.937,         # Momentum gradien (mencegah training tersendat di lokal minima)
    # weight_decay=0.0005,    # Penalti kompleksitas model (mencegah overfitting)
    # warmup_epochs=3.0,      # Jumlah epoch untuk "pemanasan" lr dari rendah ke normal
    # warmup_momentum=0.8,    # Momentum awal saat fase pemanasan
    patience=20,            # Early Stopping: berhenti jika metrik tidak membaik selama X epoch
    
    # ==========================================
    # C. PARAMETER AUGMENTASI (SANGAT KRITIS UNTUK TEKS)
    # ==========================================
    # 1. Augmentasi Haram (WAJIB 0.0 karena objek adalah angka/teks)
    fliplr=0.0,             # JANGAN DIBALIK HORIZONTAL (73 -> E7)
    flipud=0.0,             # JANGAN DIBALIK VERTIKAL (6 -> 9)
    mixup=0.0,              # JANGAN DITUMPUK (Angka berbayang merusak bounding box)
    copy_paste=0.0,         # Tidak relevan untuk data teks utuh
    
    # 2. Augmentasi Esensial (Bisa Anda tuning nilainya)
    hsv_h=0.015,            # Variasi Warna (Hue). Kecil saja karena warna monitor medis baku
    hsv_s=0.7,              # Variasi Saturasi. (Membantu model tahan terhadap layar pudar)
    hsv_v=0.4,              # Variasi Kecerahan (Value). (Membantu model tahan cahaya silau/gelap)
    degrees=5.0,            # Rotasi maksimal gambar (derajat). 5 sudah cukup untuk kemiringan foto.
    translate=0.1,          # Geser gambar secara vertikal/horizontal (10%).
    scale=0.5,              # Zoom in/out gambar (+- 50%). Membantu mendeteksi monitor dari jauh.
    shear=2.0,              # Tarikan distorsi trapesium (2 derajat).
    perspective=0.001,      # Distorsi 3D ringan (mensimulasikan foto dari sudut miring).
    mosaic=0.0,             # Menggabungkan 4 gambar. (Standar 1.0, ubah ke 0.0 jika mAP memburuk).
    erasing=0.0,            # Random Erasing (menghapus kotak kecil acak). Simulasikan layar tertutup kabel/tangan.

    # ==========================================
    # D. PARAMETER SISTEM & REPRODUCIBILITY
    # ==========================================
    save=True,              # Simpan bobot (weights) model terbaik dan terakhir
    save_period=-1,         # Simpan checkpoint setiap X epoch (-1 artinya hanya best dan last)
    cache=False,            # Simpan data di RAM ('ram' atau 'disk'). False saja jika RAM Kaggle terbatas.
    deterministic=True,     # Memaksa hasil training bisa diulang (reproducible) dengan seed yang sama
    seed=42                 # Angka seed untuk mengunci nilai acak awal
)

# 4. Validasi Model Terbaik
print("Pelatihan selesai. Memulai validasi...")
metrics = model.val()
print(f"mAP50: {metrics.box.map50}")

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Skripsi_V1-2/data.yaml, degrees=5.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=baseline_y

### 3.2 - Save train results

In [9]:
# !zip -r yolov8_train.zip /content/runs/detect/train*

### 3.3 - Examining train results  
**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [10]:
# !ls {HOME}/runs/detect/train/
# train_folder = f'{HOME}/runs/detect/train' # Change this folder according to your recent train folder name
# weight_folder = f'{train_folder}/weights'

In [11]:
# # Confusion matrix
# from IPython.display import Image as IPyImage

# IPyImage(filename=f'{train_folder}/confusion_matrix.png', width=600)

In [12]:
# # Train graph results
# IPyImage(filename=f'{train_folder}/results.png', width=600)

In [13]:
# Validation image batch results
# IPyImage(filename=f'{train_folder}/val_batch0_pred.jpg', width=600)

## Validate fine-tuned model

In [14]:
# !yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

## Inference with custom model

In [15]:
# !yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

**NOTE:** Let's take a look at few results.

In [16]:
# import glob
# import os
# from IPython.display import Image as IPyImage, display

# latest_folder = max(glob.glob(f'{HOME}/runs/detect/predict*/'), key=os.path.getmtime)
# for img in glob.glob(f'{latest_folder}/*.jpg'):
#     display(IPyImage(filename=img, width=600))
#     print("\n")